# Trace Count v5.2: switch-token diagnostics

目标：解释为什么 mixed thinking-toggle Transformer 里，开关看起来学会了，但 targeted retrieval 不显著。

核心检查：

1. `<Think/>` / `</Think>` 的 switch logits 是否把 thinking 与 non-thinking 分开；
2. retrieval 是否测在正确的 **prediction query** 上，而不是 marker token 已经可见之后；
3. marker-only trace 是否因为缺少显式 `index_token_k`，天然不容易形成 v2 那种 k-to-k retrieval head。
            

## 1. Setup

In [1]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
INSTALL_DEPS = False
PULL_REPO = True

if IN_COLAB:
    repo_dir = Path("/content/Synthetic_CoT_NiaH_Count")
    cwd = Path.cwd()
    if (cwd / ".git").exists() or (cwd / "README.md").exists():
        repo_dir = cwd
    elif not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    if PULL_REPO and (repo_dir / ".git").exists():
        subprocess.run(["git", "pull", "--ff-only"], check=False)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "pandas", "matplotlib", "tqdm"], check=True)

import pandas as pd
from IPython.display import Markdown, display, Image

display(Markdown(f"**Repo root:** `{ROOT}`"))

**Repo root:** `/content/Synthetic_CoT_NiaH_Count`

## 2. Runtime settings

In [2]:
# A completed v5 checkpoint is required; v5.2 does not retrain the model.
# Set this only when you want to force a particular run or result bundle.
RUN_DIR_OVERRIDE = ""
AUTO_MOUNT_DRIVE_FOR_V5 = True
PREFER_DRIVE_V5 = True
DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)

from synthetic_counting_extensions.v5_2_switch_diagnostics import resolve_v5_run_dir


def find_v5_run() -> Path:
    if RUN_DIR_OVERRIDE:
        return resolve_v5_run_dir(RUN_DIR_OVERRIDE)

    local_search_roots = [
        Path("outputs/v5"),
        Path("outputs"),
        Path("runs"),
        Path("colab_results"),
    ]
    drive_search_roots = [DRIVE_RESULTS_ROOT / "v5", DRIVE_RESULTS_ROOT]

    if IN_COLAB and PREFER_DRIVE_V5 and AUTO_MOUNT_DRIVE_FOR_V5:
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive

            drive.mount("/content/drive")
        for root in drive_search_roots:
            try:
                return resolve_v5_run_dir(root)
            except FileNotFoundError:
                pass

    for root in local_search_roots:
        try:
            return resolve_v5_run_dir(root)
        except FileNotFoundError:
            pass

    if IN_COLAB and not PREFER_DRIVE_V5 and AUTO_MOUNT_DRIVE_FOR_V5:
        if not Path("/content/drive/MyDrive").exists():
            from google.colab import drive

            drive.mount("/content/drive")
        for root in drive_search_roots:
            try:
                return resolve_v5_run_dir(root)
            except FileNotFoundError:
                pass

    searched = [str(path) for path in local_search_roots]
    if IN_COLAB:
        searched.append(str(DRIVE_RESULTS_ROOT))
    raise FileNotFoundError(
        "No completed v5 run was found. Run Trace_Count_v5_Colab first, or set "
        "RUN_DIR_OVERRIDE to a saved v5 result folder. Searched: " + ", ".join(searched)
    )


RUN_DIR = find_v5_run()

EXAMPLES_PER_COUNT = 100
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print({
    "RUN_DIR": str(RUN_DIR),
    "CHECKPOINT": str(RUN_DIR / "checkpoints" / "final.pt"),
    "EXAMPLES_PER_COUNT": EXAMPLES_PER_COUNT,
    "DEVICE": DEVICE,
})
            

ImportError: cannot import name 'resolve_v5_run_dir' from 'synthetic_counting_extensions.v5_2_switch_diagnostics' (/content/Synthetic_CoT_NiaH_Count/synthetic_counting_extensions/v5_2_switch_diagnostics.py)

## 3. Run v5.2 diagnostics

In [ ]:
from synthetic_counting_extensions.v5_2_switch_diagnostics import run_switch_and_retrieval_diagnostics

outputs = run_switch_and_retrieval_diagnostics(RUN_DIR, examples_per_count=EXAMPLES_PER_COUNT, device=DEVICE)
display(Markdown(f"**Report:** `{RUN_DIR / 'v5_2_switch_diagnostics' / 'report' / 'report.html'}`"))
display(outputs["switch_summary"])
display(outputs["prediction_query_head_summary"].sort_values(["correct_top1", "correct_prompt_needle_mass"], ascending=False).head(12))
            

## 4. Display figures

In [ ]:
FIG_DIR = RUN_DIR / "v5_2_switch_diagnostics" / "figures"
for name in [
    "switch_probability_summary.png",
    "prediction_query_correct_top1.png",
    "prediction_query_correct_mass.png",
    "prediction_query_marker_margin.png",
    "post_marker_correct_top1.png",
]:
    path = FIG_DIR / name
    if path.exists():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))
            

## 5. Save / GitHub / disconnect

In [ ]:
# Save result folder to Google Drive.
SAVE_TO_DRIVE = True
DRIVE_DEST_ROOT = "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results"

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_root = Path(DRIVE_DEST_ROOT)
    dest_root.mkdir(parents=True, exist_ok=True)
    if "RUN_DIR" in globals() and RUN_DIR is not None:
        src = Path(RUN_DIR)
    elif "OUT_ROOT" in globals():
        src = Path(OUT_ROOT)
    else:
        src = None
    if src is not None and src.exists():
        dest = dest_root / src.name
        src_resolved = src.resolve()
        dest_root_resolved = dest_root.resolve()
        if src_resolved == dest_root_resolved or dest_root_resolved in src_resolved.parents:
            display(Markdown(f"**Already stored in Drive:** `{src}`"))
        else:
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(src, dest)
            display(Markdown(f"**Saved to Drive:** `{dest}`"))
    else:
        display(Markdown("No RUN_DIR/OUT_ROOT found to save."))
else:
    display(Markdown("Drive save skipped."))

In [ ]:
# Optional: commit and push notebook/code changes to GitHub.
PUSH_TO_GITHUB = False
GIT_COMMIT_MESSAGE = "Add synthetic counting experiment notebook"

if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=False)
    subprocess.run(["git", "add", "notebooks", "synthetic_counting_extensions", "scripts"], check=True)
    subprocess.run(["git", "commit", "-m", GIT_COMMIT_MESSAGE], check=True)
    subprocess.run(["git", "push"], check=True)
else:
    display(Markdown("GitHub push skipped. Set `PUSH_TO_GITHUB = True` after checking the diff."))

In [ ]:
# Optional: disconnect Colab runtime after saving.
AUTO_DISCONNECT_AFTER_SAVE = False

if AUTO_DISCONNECT_AFTER_SAVE and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()
else:
    display(Markdown("Auto-disconnect skipped."))